# 基于高阶API的矩阵乘算子开发

前面我们学习了Ascend C矩阵编程的相关基础知识，本章将在前文基础上，完整呈现如何基于高阶API实现矩阵乘Ascend C算子开发。

本节学习大纲如下：

- 算子分析
- 算子规格描述
- 创建算子工程
- Matmul算子开发

---

## 1. 环境准备
正式开始学习之前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，同时完成了代码目录的创建。保证能正常导入代码以及使用bisheng编译器，完成算子的开发及编译。

In [ ]:
!mkdir -p Sources/04.03

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

---

## 2. 算子分析：

### 2.1 明确算子的数学表达式及计算逻辑。

Matmul算子的计算公式为：

C = A * B + Bias，其示意图如下。

<img src="./images/matmul_matrix_multiplication_diagram.png" alt="Matmul矩阵乘示意图"  width="700px" >

- A、B为源操作数，A为左矩阵，形状为[M, K]；B为右矩阵，形状为[K, N]。
- C为目的操作数，存放矩阵乘结果的矩阵，形状为[M, N]。
- Bias为矩阵乘偏置，形状为[1, N]。对A*B结果矩阵的每一行都采用该Bias进行偏置。

使用矩阵乘高阶API编程的计算逻辑是数据需经历“搬运-计算-搬运”三个阶段：
- **CopyIn阶段**对应`SetTensorA`、`SetTensorB`、`SetBias`接口。
- **Compute阶段**对应`Iterate`接口。
- **CopyOut阶段**对应`GetTensorC`接口。

### 2.2 明确输入和输出。
- 输入数量：3个，分别为 a、b、bias。
- 输出数量：1个，为 c。
- 数据类型：输入a、b为half类型，bias与输出为float类型。
- 数据形状（Shape）： 输入a Shape为 (M, K)，输入b Shape为 (K, N), 输入bias Shape为 (1, N), 输出c Shape为（M, N）。此处仅以M = 1024， K = 256， N = 640输入数据作为演示示例，在真实业务场景下，可灵活调整输入数据的shape参数，也可直接支持泛化输入形式。
- 数据布局（Format）：ND。

### 2.3 确定核函数名称和参数。
根据输入输出分析，确定核函数参数，定义核函数原型：
- 核函数名称：matmul_custom
- 参数列表：
- a, b, bias: 输入数据在Global Memory上的起始地址。
- c: 输出数据在Global Memory上的起始地址。

---

## 3. 算子规格描述

通过以上分析，得到Ascend C Matmul算子的设计规格如下：


<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 550px; border: 1px solid #ddd;">
  <!-- 表头行：算子类型 -->
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子类型 (OpType)</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" > MatmulCustom</td>
  </tr>
  
  <!-- 算子输入模块：纵向合并单元格 -->
  <tr>
    <!-- 算子输入 纵向合并3行（x行+y行+列名行） -->
    <td rowspan="4" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">shape</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">a</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(1024, 256)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">b</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(256, 640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
   <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">bias</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr> 
  <!-- 算子输出模块：纵向合并单元格 -->

  <tr>
  <td  style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">c</td>
    <td style="padding: 8px;">(1024, 640)</td>
    <td style="padding: 8px;">float</td>
    <td style="padding: 8px;">ND</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>  

---

## 4. 创建算子工程

### 4.1 编写Matmul算子的原型定义json文件

MatmulCustom算子的json文件命名为MatmulCustom.json，文件内容如下：

In [ ]:
%%writefile Sources/04.03/MatmulCustom.json

[
    {
        "op": "MatmulCustom",
        "language": "cpp",
        "input_desc": [
            {
                "name": "a",
                "param_type": "required",
                "format": [
                    "ND"
                ],
                "type": [
                    "float16"
                ]
            },
            {
                "name": "b",
                "param_type": "required",
                "format": [
                    "ND"
                ],
                "type": [
                    "float16"
                ]
            },
            {
                "name": "bias",
                "param_type": "required",
                "format": [
                    "ND"
                ],
                "type": [
                    "float"
                ]
            }
        ],
        "output_desc": [
            {
                "name": "c",
                "param_type": "required",
                "format": [
                    "ND"
                ],
                "type": [
                    "float"
                ]
            }
        ]
    }
]

### 4.2 使用msopgen工具生成Matmul算子工程

执行以下命令安装msOpGen工具：

In [ ]:
# 下载工具仓代码
!git clone https://gitcode.com/Ascend/msopgen.git
#编译安装msopgen工具
!cd msopgen; python build.py;cd output;pip install mindstudio_opgen*.whl --force-reinstall;pip install mindstudio_opst*.whl --force-reinstall


准备好msopgen工具后，执行如下命令会在指定目录下生成算子工程，工程中包含算子实现的模板文件，编译脚本等。：

In [ ]:
# 清除已有的MatmulCustom目录
!rm -rf Sources/04.03/MatmulCustom
# 使用msopgen工具创建算子工程前设置使用开源仓编译的msopgen工具，不设置时默认使用CANN安装目录下的msopgen工具
!export PATH=/usr/local/bin:~/.local/bin:$PATH;msopgen gen -i Sources/04.03/MatmulCustom.json -c ai_core-ascend910b1 -lan cpp -out Sources/04.03/MatmulCustom

---

## 5. Matmul算子开发

在[矩阵乘法介绍章节](./04.02_matrix_multiplication_introduction.ipynb)介绍了Matmul矩阵乘的数据切分方案和数据流。Ascend C提供一组Matmul高阶API，封装了这些常用的切分和数据搬运、计算的算法逻辑，方便开发者快速实现Matmul矩阵乘法的运算操作。如下图所示，开发者在host侧通过调用API自动获取Tiling参数，该参数传递到kernel侧后，在初始化操作时传入，通过几个简单的API即可完成矩阵乘操作。

<img src="./images/process.png" alt="开发过程"  width="700px" >

算子工程中通过修改如下文件完成算子计算相关的开发适配，包括：
- op_host/matmul_custom_tiling.h，算子tiling结构体定义
- op_host/matmul_custom.cpp，矩阵乘tiling算法实现
- op_kernel/matmul_custom.cpp，算子核函数的开发

### 5.1 tiling结构体定义

在算子工程中op_host/matmul_custom_tiling.h定义了 MatmulCustom 算子的 Tiling 数据结构，用于在主机侧与设备侧之间传递矩阵乘法的切分参数。

在MatmulCustomTilingData结构体中，我们添加如下两个字段：

1. **定义uint64_t类型的字段localMemSize**，用于存储当前核所需的本地内存（UB）大小。该值在主机侧根据平台信息设置，核函数中通过 tilingData.localMemSize 获取。
2. **定义TCubeTiling类型的字段cubeTilingData**，TCubeTiling是 Matmul 高阶 API 预定义的切分参数结构体，包含 M、N、Ka、Kb、singleCoreM/N/K、baseM/N/K 等核心切分信息。核函数通过该字段获取完整的矩阵乘法切分配置,用于指导后续的数据切块、搬运及矩阵计算过程。TCubeTiling结构体的主要参数说明如下表所示。

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center"><strong>参数名称</strong></td>
  <td align="center"><strong>数据类型</strong></td>
  <td align="center"><strong>说明</strong></td>
</tr>
<tr>
  <td align="left">usedCoreNum</td>
  <td align="left">int</td>
  <td align="left">使用的AI处理器核数，请根据实际情况设置。取值范围为：[1, AI处理器最大核数]。<br><br>该参数与shape相关参数的关系为：usedCoreNum = (M / singleCoreM) * (N / singleCoreN)。<br><br></td>
</tr>
<tr>
  <td align="left">M, N, Ka, Kb</td>
  <td align="left">int</td>
  <td align="left">A、B、C矩阵原始输入的shape大小，以元素为单位。<br><br>M, Ka为A矩阵原始输入的Shape，Kb, N为B矩阵原始输入的Shape。<br><br></td>
</tr>
<tr>
  <td align="left">singleCoreM, singleCoreN, singleCoreK</td>
  <td align="left">int</td>
  <td align="left">A、B、C矩阵单核内shape大小，以元素为单位。该参数取值必须大于0。<br><br>singleCoreK = K，多核处理时不对K进行切分；singleCoreM <= M；singleCoreN <= N。<br><br> </td>
</tr>
<tr>
  <td align="left">baseM, baseN, baseK</td>
  <td align="left">int</td>
  <td align="left">A、B、C矩阵参与一次矩阵乘指令的shape大小，以元素为单位。<br><br>A、B、C矩阵参与一次矩阵乘的shape大小需要按分形对齐。<br><br> </td>
</tr>
<tr>
  <td align="left">depthA1, depthB1</td>
  <td align="left">int</td>
  <td align="left">A1、B1中全载基本块的份数，depthA1为A1中全载baseM * baseK的份数，depthB1为B1中全载baseN * baseK的份数。<br><br></td>
</tr>
<tr>
  <td align="left">stepM, stepN, stepKa, stepKb</td>
  <td align="left">int</td>
  <td align="left">stepM为左矩阵在A1中缓存的buffer M方向上baseM的倍数。<br>stepN为右矩阵在B1中缓存的buffer N方向上baseN的倍数。<br>stepKa为左矩阵在A1中缓存的buffer Ka方向上baseK的倍数。<br>stepKb为右矩阵在B1中缓存的buffer Kb方向上baseK的倍数。<br><br></td>
</tr>
<tr>
  <td align="left">isBias</td>
  <td align="left">int</td>
  <td align="left">是否使能Bias，参数取值如下：<br>0：不使能Bias（默认值）。<br>1：使能Bias。<br></td>
</tr>
<tr>
  <td align="left">iterateOrder</td>
  <td align="left">int</td>
  <td align="left">一次Iterate计算出[baseM, baseN]大小的C矩阵分片，Iterate完成后，Matmul会自动偏移下一次Iterate输出的C矩阵位置，iterOrder表示自动偏移的顺序。参数取值如下：<br>0：先往M轴方向偏移再往N轴方向偏移。<br>1：先往N轴方向偏移再往M轴方向偏移。<br></td>
</tr>
</table>


In [ ]:
%%writefile Sources/04.03/MatmulCustom/op_host/matmul_custom_tiling.h

#include "register/tilingdata_base.h"
#include "tiling/tiling_api.h"

namespace optiling {
BEGIN_TILING_DATA_DEF(MatmulCustomTilingData)
  TILING_DATA_FIELD_DEF(uint64_t, localMemSize);
  TILING_DATA_FIELD_DEF_STRUCT(TCubeTiling, cubeTilingData);
END_TILING_DATA_DEF;

REGISTER_TILING_DATA_CLASS(MatmulCustom, MatmulCustomTilingData)
}

### 5.2 矩阵乘tiling算法实现

op_host/matmul_custom.cpp文件位于算子主机侧（op_host）目录，是 MatmulCustom 算子的主机侧逻辑实现，负责以下两个核心任务：

- **Tiling 函数实现**

    根据输入张量的实际形状、数据类型及当前硬件平台特性，动态计算矩阵乘法所需的切分参数（如单核计算尺寸、循环步长、基础块大小等），并将这些参数填充至预定义的 Tiling 数据结构中，供设备侧核函数运行时解析和使用。

- **算子原型定义**

    通过原型定义来描述算子输入输出、属性等信息以及算子在AI处理器上相关实现信息，并关联tiling实现等函数。

在正式实现矩阵乘算子的Tiling函数之前，有必要先了解 Ascend C 提供的 Matmul Tiling API 的基本概念及其获取方式。本节将对此进行详细说明。

#### 5.2.1 Matmul Tiling API介绍

Ascend C提供一组Matmul Tiling API，方便用户获取Matmul kernel计算时所需的Tiling参数。用户只需要传入A/B/C矩阵的Position位置、Format格式和DType数据类型等信息，调用API接口，即可获取到TCubeTiling结构体中的相关参数。

Matmul Tiling API分为Matmul单核Tiling接口、多核Tiling接口和BatchMatmul Tiling接口，本章基于多核并行场景，重点介绍多核 Tiling 接口 —— MultiCoreMatmulTiling，其详细说明如下表所示。

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center"><strong>接口</strong></td>
  <td align="center"><strong>功能</strong></td>
</tr>
<tr>
  <td align="left">SetAType</td>
  <td align="left">设置A矩阵的位置，数据格式，数据类型，是否转置等信息。</td>
</tr>
<tr>
  <td align="left">SetBType</td>
  <td align="left">设置B矩阵的位置，数据格式，数据类型，是否转置等信息。</td>
</tr>
<tr>
  <td align="left">SetCType</td>
  <td align="left">设置C矩阵的位置，数据格式，数据类型等信息。</td>
</tr>
<tr>
  <td align="left">SetBiasType</td>
  <td align="left">设置Bias的位置，数据格式，数据类型等信息。</td>
</tr>
<tr>
  <td align="left">SetShape</td>
  <td align="left">设置Matmul单次计算的形状singleM、singleN、singleK，单位为元素个数。</td>
</tr>
<tr>
  <td align="left">SetOrgShape</td>
  <td align="left">设置Matmul计算时的原始完整的形状M、N、Ka、Kb，单位为元素个数。</td>
</tr>
<tr>
  <td align="left">SetBatchInfoForNormal</td>
  <td align="left">设置A/B矩阵的M/N/K轴信息，以及A/B矩阵各自的Batch数。</td>
</tr>
<tr>
  <td align="left">SetFixSplit</td>
  <td align="left">设置固定的baseM、baseN、baseK，单位为元素个数。</td>
</tr>
<tr>
  <td align="left">SetBufferSpace</td>
  <td align="left">设置Matmul计算时可用的L1/L0C/UB空间大小，单位为字节。</td>
</tr>
<tr>
  <td align="left">SetTraverse</td>
  <td align="left">设置遍历方式，M轴优先还是N轴优先。</td>
</tr>
<tr>
  <td align="left">GetBaseM</td>
  <td align="left">获取baseM值。</td>
</tr>
<tr>
  <td align="left">GetBaseN</td>
  <td align="left">获取baseN值。</td>
</tr>
<tr>
  <td align="left">GetBaseK</td>
  <td align="left">获取baseK值。</td>
</tr>
<tr>
  <td align="left">GetTiling</td>
  <td align="left">获取Tiling参数。</td>
</tr>
<tr>
  <td align="left">SetDim</td>
  <td align="left">设置多核Matmul时，可以参与运算的核数。</td>
</tr>
<tr>
  <td align="left">SetSingleRange</td>
  <td align="left">设置singleCoreM/singleCoreN/singleCoreK的最大值与最小值，单位为元素个数。</td>
</tr>
<tr>
  <td align="left">SetSingleShape</td>
  <td align="left">设置Matmul单核计算的形状singleCoreM、singleCoreN、singleCoreK，单位为元素个数。</td>
</tr>
<tr>
  <td align="left">GetSingleShape</td>
  <td align="left">获取计算后的singleCoreM/singleCoreN/singleCoreK。</td>
</tr>
<tr>
  <td align="left">SetAlignSplit</td>
  <td align="left">设置多核切分时singleCoreM/singleCoreN/singleCoreK的对齐值</td>
</tr>
<tr>
  <td align="left">GetCoreNum</td>
  <td align="left">获得多核切分后， 使用的blockDim。</td>
</tr>
</table>

#### 5.2.2 如何获取Matmul Tiling参数

本节以多核并行场景为例，介绍如何通过 MultiCoreMatmulTiling 接口获取矩阵乘所需的 Tiling 参数，具体流程如下：

1. **创建多核 Tiling 对象**

    实例化 MultiCoreMatmulTiling 对象，并传入平台信息。

2. **配置矩阵属性与切分策略**

    设置 A、B、C、Bias 的数据类型、存储位置及数据格式；同时指定矩阵的 M、N、K 形状信息，并可选配基础分块大小（baseM/baseN）、核数等参数。

3. **生成 Tiling 参数**

    调用 GetTiling 接口，将计算得到的切分参数输出至 TCubeTiling 结构体中，供后续核函数使用。

#### 5.2.3 基于 Matmul Tiling API 实现 Tiling 函数

以下代码展示了如何基于 MultiCoreMatmulTiling 接口实现矩阵乘算子的 Tiling 函数，完整演示了多核 Tiling 参数的获取与填充过程：

In [ ]:
%%writefile Sources/04.03/MatmulCustom/op_host/matmul_custom.cpp

// 包含自定义Tiling数据结构头文件，定义了MatmulCustomTilingData
#include "matmul_custom_tiling.h"
// 昇腾算子注册接口，用于定义算子属性、输入输出等
#include "register/op_def_registry.h"
// 昇腾平台信息获取接口，用于查询芯片版本、内存大小等
#include "tiling/platform/platform_ascendc.h"
// 矩阵乘Tiling核心API，包含MultiCoreMatmulTiling等
#include "tiling/tiling_api.h"
// 使用matmul_tiling命名空间，简化类型名
using namespace matmul_tiling;

namespace optiling {

// 动态生成矩阵乘法所需的切分参数。
static ge::graphStatus TilingFunc(gert::TilingContext* context)
{
    // -------------------- 步骤1：创建多核Tiling对象 --------------------
    // 1.1 获取当前运行环境的平台信息（芯片型号等）
    auto ascendcPlatform = platform_ascendc::PlatformAscendC(context->GetPlatformInfo());
    
    // 1.2 设置矩阵乘法输入shape形状
    int32_t M = 1024;
    int32_t N = 640;      // B的列数
    int32_t K = 256;      // A的列数 / B的行数
    
    // 1.3 指定基础分块大小（Cube单次指令执行的计算量），这里固定为128x128
    int32_t baseM = 128;
    int32_t baseN = 128;
    
    // 1.4 实例化多核Tiling对象，传入平台信息
    MultiCoreMatmulTiling cubeTiling(ascendcPlatform);
    
    // -------------------- 步骤2：配置矩阵属性与切分策略 --------------------
    // 2.1 设置参与多核并行计算的AI Core数量（必须与后续SetBlockDim一致）
    cubeTiling.SetDim(2);   // 使用2个核并行计算
    
    // 2.2 配置A矩阵：存储位置GM、数据格式ND、数据类型FP16
    cubeTiling.SetAType(TPosition::GM, CubeFormat::ND, matmul_tiling::DataType::DT_FLOAT16);
    // 2.3 配置B矩阵：同样为GM、ND、FP16
    cubeTiling.SetBType(TPosition::GM, CubeFormat::ND, matmul_tiling::DataType::DT_FLOAT16);
    // 2.4 配置C矩阵：输出为GM、ND、FP32
    cubeTiling.SetCType(TPosition::GM, CubeFormat::ND, matmul_tiling::DataType::DT_FLOAT);
    // 2.5 配置Bias：GM、ND、FP32
    cubeTiling.SetBiasType(TPosition::GM, CubeFormat::ND, matmul_tiling::DataType::DT_FLOAT);
    
    // 2.6 设置矩阵实际形状（用于计算）
    cubeTiling.SetShape(M, N, K);
    // 2.7 设置原始形状（用于尾块处理，此处与实际形状相同）
    cubeTiling.SetOrgShape(M, N, K);
    
    // 2.8 固定基础分块尺寸：baseM=128，baseN=128，baseK=-1表示由库自动选择
    cubeTiling.SetFixSplit(baseM, baseN, -1);
    
    // 2.9 启用Bias计算
    cubeTiling.SetBias(true);
    
    // 2.10 设置L1缓冲区分配比例，-1表示由Tiling库自动分配
    cubeTiling.SetBufferSpace(-1, -1, -1);
    
    // -------------------- 步骤3：生成Tiling参数 --------------------
    // 3.1 实例化自定义Tiling数据结构，用于接收生成的参数
    MatmulCustomTilingData tiling;
    
    // 3.2 调用GetTiling接口，将计算好的切分参数填充到tiling.cubeTilingData中
    if (cubeTiling.GetTiling(tiling.cubeTilingData) == -1) {
        return ge::GRAPH_FAILED;   // Tiling生成失败
    }
    
    // 3.3 查询当前平台每个AI Core的Unified Buffer（UB）大小（字节）
    uint64_t localMemSize;
    ascendcPlatform.GetCoreMemSize(platform_ascendc::CoreMemType::UB, localMemSize);
    tiling.set_localMemSize(localMemSize);   // 存入Tiling数据，供核函数分配工作空间
    
    // 设置执行算子的CUBE核数
    context->SetBlockDim(1);
    
    // -------------------- 将Tiling数据写入context --------------------
    // 序列化Tiling对象，存入context，供核函数通过GET_TILING_DATA宏解析
    tiling.SaveToBuffer(context->GetRawTilingData()->GetData(),
                        context->GetRawTilingData()->GetCapacity());
    context->GetRawTilingData()->SetDataSize(tiling.GetDataSize());
    
    // -------------------- 设置工作空间大小 --------------------
    size_t userWorkspaceSize = 0;
    // 系统工作空间：Matmul高阶API内部需要的临时内存，通过平台接口获取
    size_t systemWorkspaceSize = static_cast<size_t>(ascendcPlatform.GetLibApiWorkSpaceSize());
    size_t *currentWorkspace = context->GetWorkspaceSizes(1);
    currentWorkspace[0] = userWorkspaceSize + systemWorkspaceSize;
    
    return ge::GRAPH_SUCCESS;
}

} // namespace optiling

namespace ge {

static ge::graphStatus InferShape(gert::InferShapeContext* context)
{
    return GRAPH_SUCCESS;
}

static ge::graphStatus InferDataType(gert::InferDataTypeContext *context)
{          
    return ge::GRAPH_SUCCESS;
}

} // namespace ge

namespace ops {

class MatmulCustom : public OpDef {
public:
    explicit MatmulCustom(const char* name) : OpDef(name)
    {
        this->Input("a")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16})
            .Format({ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND});
        this->Input("b")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16})
            .Format({ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND});
        this->Input("bias")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT})
            .Format({ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND});
        this->Output("c")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT})
            .Format({ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND});
        this->SetInferShape(ge::InferShape)
            .SetInferDataType(ge::InferDataType);
        this->AICore()
            .SetTiling(optiling::TilingFunc);   // 设置Tiling函数
        this->AICore().AddConfig("ascend910b");
    }
};

OP_ADD(MatmulCustom);

} // namespace ops

### 5.3 算子核函数的开发

算子核函数位于 op_kernel/matmul_custom.cpp 文件中，运行于设备侧（AI Core），负责执行实际的矩阵乘法计算。Ascend C 提供的 Matmul 高阶 API 封装了格式转换、数据分块、流水并行等底层细节，开发者只需遵循固定的调用流程即可快速实现高性能矩阵乘。

在上一章节，我们介绍了使用 Matmul 高阶 API 开发核函数的五个标准步骤，并深入剖析了多核数据切分与单核内的迭代原理。接下来，我们将结合 op_kernel/matmul_custom.cpp 中的代码实例，具体演示如何将这些步骤和原理落地实现。

#### 5.3.1 准备工作

- 包含 Ascend C 核函数基础头文件与 Matmul 高阶 API 接口。
- Ceiling 函数整数除法向上取整，用于后续的分块数量计算。

In [ ]:
%%writefile Sources/04.03/MatmulCustom/op_kernel/matmul_custom.cpp

#include "kernel_operator.h"
#include "lib/matmul_intf.h"

using namespace matmul;

// 辅助函数：整数除法向上取整，用于分块数量计算
__aicore__ inline uint32_t Ceiling(uint32_t a, uint32_t b)
{
    return (a + b - 1) / b;
}

#### 5.3.2 创建核函数类

这里我们首先创建了核函数类MatmulKernel，内部函数的调用关系如下所示：

<img src="./images/matmul_kernel_flow_chart.png" alt="matmulKernel流程图"  width="700px" >

- Init函数完成初始化, 通过CalcOffset函数计算单个AICore处理数据段起始位置。
- Process中通过矩阵高阶API完成矩阵乘法运算。
- 创建Matmul对象**matmulObj**，并通过MatmulType指定输入输出矩阵的数据类型、存储位置与数据格式。
- GlobalTensor 是昇腾设备侧用于表示位于Global Memory张量的数据类型，支持切片和偏移操作。
- tiling用于保存主机侧传递的切分参数。
- localMemSize保存当前平台的UB大小。

In [ ]:
%%writefile -a Sources/04.03/MatmulCustom/op_kernel/matmul_custom.cpp

template <typename aType, typename bType, typename cType, typename biasType> 
class MatmulKernel {
public:
    __aicore__ inline MatmulKernel(){};
    __aicore__ inline void Init(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c, GM_ADDR workspace,
                                uint64_t memSize, const TCubeTiling &tiling);
    template <bool setTmpSpace = false> __aicore__ inline void Process(AscendC::TPipe *pipe);

    __aicore__ inline void CalcOffset(int32_t blockIdx, const TCubeTiling &tiling, int32_t &offsetA, int32_t &offsetB,
                                      int32_t &offsetC, int32_t &offsetBias);

    // Matmul 高阶 API 核心对象，模板参数描述 A/B/C/Bias 的类型信息：
    // - 内存位置：GM
    // - 数据格式：ND（行优先，API内部自动转换为分形格式）
    // - 数据类型：由模板参数 aType/bType/cType/biasType 指定
    Matmul<MatmulType<AscendC::TPosition::GM, CubeFormat::ND, aType>,
           MatmulType<AscendC::TPosition::GM, CubeFormat::ND, bType>,
           MatmulType<AscendC::TPosition::GM, CubeFormat::ND, cType>,
           MatmulType<AscendC::TPosition::GM, CubeFormat::ND, biasType>>
        matmulObj;

    // 当前核所负责的全局张量视图
    AscendC::GlobalTensor<aType> aGlobal;
    AscendC::GlobalTensor<bType> bGlobal;
    AscendC::GlobalTensor<cType> cGlobal;
    AscendC::GlobalTensor<biasType> biasGlobal;

    TCubeTiling tiling;      // 保存主机侧传递的切分参数
    uint64_t localMemSize;   // 保存当前平台的UB大小，用于工作空间分配
};

#### 5.3.3 进行核函数的定义

核函数（Kernel Function）作为 Ascend C 算子在设备侧的入口，负责接收主机侧传递的全局内存地址与 Tiling 参数。每个 AI Core 独立运行一份核函数实例，通过 GetBlockIdx() 获取当前核索引，以区分不同核的计算任务。在核函数内部，遵循以下五个标准步骤，完整串联 Matmul 高阶 API 以实现矩阵乘法算子：创建 Matmul 对象 → 初始化 Matmul 对象 → 设置输入数据（A、B、Bias）→ 执行矩阵乘法 → 结束矩阵乘法。

具体实现时，首先实例化 MatmulKernel 类对象，其中包含 Matmul 对象 matmulObj；随后通过 REGIST_MATMUL_OBJ 宏完成 Matmul 对象的初始化，；接着调用 Init 函数完成当前核的数据偏移计算；最后调用Process 函数，执行矩阵乘法的核心计算与结束清理。




In [ ]:
%%writefile Sources/04.03/MatmulCustom/op_kernel/matmul_custom_bak.cpp

extern "C" __global__ __aicore__ void matmul_custom(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c,
                                                    GM_ADDR workspace, GM_ADDR tiling)
{
    GET_TILING_DATA(tilingData, tiling);

    // ----- 步骤1（实例化）：创建 MatmulKernel 对象，内部包含 Matmul 对象 -----
    MatmulKernel<half, half, float, float> matmulKernel;

    // ----- 步骤2：初始化 Matmul 对象 -----
    AscendC::TPipe pipe;
    // REGIST_MATMUL_OBJ 宏完成 Matmul 对象与流水线、Tiling 参数的绑定
    REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), matmulKernel.matmulObj,
                      &tilingData.cubeTilingData);

    // ----- 步骤3：设置输入数据（初始化 + 偏移切片）-----
    matmulKernel.Init(a, b, bias, c, workspace,
                      tilingData.localMemSize, tilingData.cubeTilingData);

    // ----- 步骤4 & 5：完成计算与清理 -----
    matmulKernel.Process(&pipe);
}

#### 5.3.4 Init函数实现

初始化函数Init主要完成以下内容：

- 根据核索引计算当前核在全局矩阵中的偏移量（CalcOffset）。
- 设置当前核对应输入输出Global Tensor的Global Memory内存地址。

In [ ]:
%%writefile -a Sources/04.03/MatmulCustom/op_kernel/matmul_custom.cpp

template <typename aType, typename bType, typename cType, typename biasType>
__aicore__ inline void
MatmulKernel<aType, bType, cType, biasType>::CalcOffset(int32_t blockIdx, const TCubeTiling &tiling,
                                                        int32_t &offsetA, int32_t &offsetB,
                                                        int32_t &offsetC, int32_t &offsetBias)
{
    // M 维度上可划分的单核块数（向上取整，处理尾块）
    auto mSingleBlocks = Ceiling(tiling.M, tiling.singleCoreM);
    // 当前核在 M 方向上的索引（0 ~ mSingleBlocks-1）
    auto mCoreIndx = blockIdx % mSingleBlocks;
    // 当前核在 N 方向上的索引
    auto nCoreIndx = blockIdx / mSingleBlocks;

    // A 矩阵形状 [M, Ka]，行优先存储：偏移 = 起始行号 × Ka
    offsetA = mCoreIndx * tiling.Ka * tiling.singleCoreM;
    // B 矩阵形状 [Kb, N]，行优先存储：偏移 = 起始列号（单核负责连续 N 方向块）
    offsetB = nCoreIndx * tiling.singleCoreN;
    // C 矩阵形状 [M, N]：偏移 = 起始行号 × N + 起始列号
    offsetC = mCoreIndx * tiling.N * tiling.singleCoreM + nCoreIndx * tiling.singleCoreN;
    // Bias 向量长度 N，按 N 方向切分：偏移 = 起始列号
    offsetBias = nCoreIndx * tiling.singleCoreN;
}

template <typename aType, typename bType, typename cType, typename biasType>
__aicore__ inline void
MatmulKernel<aType, bType, cType, biasType>::Init(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c,
                                                  GM_ADDR workspace, uint64_t memSize,
                                                  const TCubeTiling &tiling)
{
    // 保存 Tiling 参数和 UB 大小
    this->tiling = tiling;
    this->localMemSize = memSize;

    // 将全局内存地址绑定到 GlobalTensor，并指定完整张量的总元素个数
    aGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ aType *>(a), tiling.M * tiling.Ka);
    bGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ bType *>(b), tiling.Kb * tiling.N);
    cGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ cType *>(c), tiling.M * tiling.N);
    biasGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ biasType *>(bias), tiling.N);

    // 计算当前核在全局矩阵中的偏移量
    int32_t offsetA = 0, offsetB = 0, offsetC = 0, offsetBias = 0;
    CalcOffset(GetBlockIdx(), tiling, offsetA, offsetB, offsetC, offsetBias);

    // 通过 GlobalTensor 的下标运算符 [offset] 获取从偏移位置开始的子张量视图
    // 该操作仅记录新地址，不涉及数据搬运
    aGlobal = aGlobal[offsetA];
    bGlobal = bGlobal[offsetB];
    cGlobal = cGlobal[offsetC];
    biasGlobal = biasGlobal[offsetBias];

    // 系统工作空间指针为空时直接返回
    if (GetSysWorkSpacePtr() == nullptr) {
        return;
    }
}

#### 5.3.5 Process函数实现

Process函数中通过调用Matmul高阶API执行如下操作，完成矩阵乘法计算：

- 设置左矩阵A、右矩阵B、Bias

    调用`SetTensorA`、`SetTensorB`、`SetBias`接口，将当前AI Core负责的左矩阵A、右矩阵B及偏置Bias的输入地址传递给Matmul对象。

- 执行矩阵乘操作
    
    调用`IterateAll`接口启动矩阵乘计算，结果直接写入输出地址cGlobal。

- 结束矩阵乘操作

    调用`End`接口释放Matmul计算资源。

In [ ]:
%%writefile -a Sources/04.03/MatmulCustom/op_kernel/matmul_custom.cpp

template <typename aType, typename bType, typename cType, typename biasType>
template <bool setTmpSpace>
__aicore__ inline void
MatmulKernel<aType, bType, cType, biasType>::Process(AscendC::TPipe *pipe)
{
    // 步骤4.1（可选）：显式分配 UB 工作空间并将其起始物理地址传入给Matmul
    if constexpr (setTmpSpace) {
        AscendC::TBuf<> tmpMMFormatUb;
        AscendC::LocalTensor<uint8_t> mmformatUb;
        pipe->InitBuffer(tmpMMFormatUb, localMemSize);     // 初始化缓冲区，大小为 localMemSize
        mmformatUb = tmpMMFormatUb.Get<uint8_t>(localMemSize); // 获取指定大小的本地张量
        matmulObj.SetLocalWorkspace(mmformatUb);           // 将 UB 空间设置给 Matmul 对象
    }

    // 步骤4.2：将当前核的输入地址传递给 Matmul 对象。
    matmulObj.SetTensorA(aGlobal);
    matmulObj.SetTensorB(bGlobal);
    matmulObj.SetBias(biasGlobal);

    // 步骤4.3：执行完整的矩阵乘法，结果直接写入 cGlobal 指向的全局内存
    matmulObj.IterateAll(cGlobal);

    // 步骤5：结束矩阵乘操作，释放Matmul计算资源
    matmulObj.End();
}

#### 5.3.6 重新写入核函数

由于核函数会调用实现类中的函数，所以需要将4.3.3中暂存于matmul_custom_bak.cpp中的核函数代码写入matmul_custom.cpp
%%writefile Sources/04.03/MatmulCustom/op_kernel/matmul_custom_bak.cpp


In [12]:
!cat Sources/04.03/MatmulCustom/op_kernel/matmul_custom_bak.cpp >> Sources/04.03/MatmulCustom/op_kernel/matmul_custom.cpp
!rm Sources/04.03/MatmulCustom/op_kernel/matmul_custom_bak.cpp

### 5.4 算子调用

完成算子Kernel、Host侧的开发后，为验证 MatmulCustom 算子工程的可用性，需要对算子工程进行编译，生成自定义算子安装包*.run。运行预先编写的单算子API调用程序，对该算子工程进行测试，检查其能否正常执行。具体操作步骤如下：

In [ ]:
#编译算子工程并部署编译出的算子包
!cd Sources/04.03/MatmulCustom;bash build.sh;
!./Sources/04.03/MatmulCustom/build_out/custom_opp*.run --install-path=${HOME}/

接下来编写测试代码，这里以固定的M = 1024， K = 256， N = 640。 a矩阵全1，b矩阵全2，bias矩阵全0.5数据进行测试。

In [ ]:
%%writefile Sources/04.03/aclnn_test.cpp

#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "aclnn/aclnn_base.h"
#include "aclnn/acl_meta.h"
#include "acl/acl_rt.h"
#include "aclnn_matmul_custom.h"

#define CHECK_ACL(expr)                                                                                 \
    do {                                                                                                \
        auto __ret = (expr);                                                                            \
        int32_t __code = static_cast<int32_t>(__ret);                                                   \
        if (__code != 0) {                                                                              \
            fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__, __code); \
        }                                                                                               \
    } while (0)

int32_t main(int32_t argc, char** argv)
{
    const int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    CHECK_ACL(aclnnInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));

    // 定义MatmulLeakyrelu的张量形状
    const std::vector<int64_t> shape_a = {1024, 256};    // 输入a形状
    const std::vector<int64_t> shape_b = {256, 640};     // 输入b形状
    const std::vector<int64_t> shape_bias = {640};       // 输入bias形状
    const std::vector<int64_t> shape_output = {1024, 640};// 输出形状

    // 计算各张量元素数量和内存大小
    const int64_t elementCount_a = shape_a[0] * shape_a[1];
    const int64_t elementCount_b = shape_b[0] * shape_b[1];
    const int64_t elementCount_bias = shape_bias[0];
    const int64_t elementCount_output = shape_output[0] * shape_output[1];
    
    const size_t bufferSize_a = elementCount_a * sizeof(aclFloat16);
    const size_t bufferSize_b = elementCount_b * sizeof(aclFloat16);
    const size_t bufferSize_bias = elementCount_bias * sizeof(float);
    const size_t bufferSize_output = elementCount_output * sizeof(float);

    // 分配输入a的设备内存并创建张量
    void* inputADeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputADeviceMem, bufferSize_a, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputA = aclCreateTensor(shape_a.data(), shape_a.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                        shape_a.data(), shape_a.size(), inputADeviceMem);

    // 分配输入b的设备内存并创建张量
    void* inputBDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputBDeviceMem, bufferSize_b, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputB = aclCreateTensor(shape_b.data(), shape_b.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                        shape_b.data(), shape_b.size(), inputBDeviceMem);

    // 分配输入bias的设备内存并创建张量
    void* inputBiasDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputBiasDeviceMem, bufferSize_bias, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputBias = aclCreateTensor(shape_bias.data(), shape_bias.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                           shape_bias.data(), shape_bias.size(), inputBiasDeviceMem);

    // 分配输出的设备内存并创建张量
    void* outputDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&outputDeviceMem, bufferSize_output, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* output = aclCreateTensor(shape_output.data(), shape_output.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                        shape_output.data(), shape_output.size(), outputDeviceMem);

    // 准备主机测试数据（沿用原始代码的aclFloatToFloat16函数）
    std::vector<aclFloat16> inputAHostData(elementCount_a, aclFloatToFloat16(1.0));
    std::vector<aclFloat16> inputBHostData(elementCount_b, aclFloatToFloat16(2.0));
    std::vector<float> inputBiasHostData(elementCount_bias, float(0.5));
    std::vector<float> outputHostData(elementCount_output, float(0.0));
    
    // 计算预期结果
    std::vector<float> goldenData(elementCount_output, float(512.5));

    // 主机数据拷贝到设备
    CHECK_ACL(aclrtMemcpy(inputADeviceMem, bufferSize_a, inputAHostData.data(),
                          bufferSize_a, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBDeviceMem, bufferSize_b, inputBHostData.data(),
                          bufferSize_b, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBiasDeviceMem, bufferSize_bias, inputBiasHostData.data(),
                          bufferSize_bias, ACL_MEMCPY_HOST_TO_DEVICE));

    // 获取Matmul算子工作空间大小和执行器
    uint64_t workspaceSize = 0;
    aclOpExecutor* executor = nullptr;
    CHECK_ACL(aclnnMatmulCustomGetWorkspaceSize(inputA, inputB, inputBias, output, &workspaceSize, &executor));

    // 分配工作空间内存
    void* workspaceDeviceMem = nullptr;
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtMalloc(&workspaceDeviceMem, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST));
    }

    // 执行Matmul算子
    CHECK_ACL(aclnnMatmulCustom(workspaceDeviceMem, workspaceSize, executor, stream));
    CHECK_ACL(aclrtSynchronizeStream(stream));

    // 设备结果拷贝回主机
    CHECK_ACL(aclrtMemcpy(outputHostData.data(), bufferSize_output, outputDeviceMem,
                          bufferSize_output, ACL_MEMCPY_DEVICE_TO_HOST));

    // 打印结果并验证
    printf("result is:\n");
    const int64_t previewCount = std::min<int64_t>(elementCount_output, 10);
    for (int64_t i = 0; i < previewCount; i++) { 
        printf("%.1f ", outputHostData[i]); 
    }
    printf("\ntest %s\n", std::equal(outputHostData.begin(), outputHostData.end(), goldenData.begin()) ? "pass" : "failed");

    // 释放资源
    aclDestroyTensor(inputA);
    aclDestroyTensor(inputB);
    aclDestroyTensor(inputBias);
    aclDestroyTensor(output);
    
    CHECK_ACL(aclrtFree(inputADeviceMem));
    CHECK_ACL(aclrtFree(inputBDeviceMem));
    CHECK_ACL(aclrtFree(inputBiasDeviceMem));
    CHECK_ACL(aclrtFree(outputDeviceMem));
    
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtFree(workspaceDeviceMem));
    }
    
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclnnFinalize());
    
    return 0;
}

编译并运行测试代码，检查输出是否与预期一致。

In [ ]:
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/04.03/aclnn_test.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/04.03/execute_op;
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;./Sources/04.03/execute_op

调用程序正常执行后会有如下打印：

<img src="./images/result_03.png" alt="结果图片03"  width="500px" >

---

## 课后实践


尝试修改代码，实现Matmul的计算。公式为：

C = A * B + Bias

支持M = 1024, N = 2048, K = 512的half类型输入，float类型输出，算子原型为：

<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 550px; border: 1px solid #ddd;">
  <!-- 表头行：算子类型 -->
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子类型 (OpType)</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" > MatmulCustom</td>
  </tr>
  
  <!-- 算子输入模块：纵向合并单元格 -->
  <tr>
    <!-- 算子输入 纵向合并3行（x行+y行+列名行） -->
    <td rowspan="4" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">shape</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">a</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(1024, 512)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">b</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(512, 2048)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
   <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">bias</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(2048)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr> 
  <!-- 算子输出模块：纵向合并单元格 -->

  <tr>
  <td  style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">c</td>
    <td style="padding: 8px;">(1024, 2048)</td>
    <td style="padding: 8px;">float</td>
    <td style="padding: 8px;">ND</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>  

### 算子原型json文件

该部分不需要修改，直接执行即可。

In [ ]:
%%writefile Sources/04.03/MatmulCustom.json

[
    {
        "op": "MatmulCustom",
        "language": "cpp",
        "input_desc": [
            {
                "name": "a",
                "param_type": "required",
                "format": [
                    "ND"
                ],
                "type": [
                    "float16"
                ]
            },
            {
                "name": "b",
                "param_type": "required",
                "format": [
                    "ND"
                ],
                "type": [
                    "float16"
                ]
            },
            {
                "name": "bias",
                "param_type": "required",
                "format": [
                    "ND"
                ],
                "type": [
                    "float"
                ]
            }
        ],
        "output_desc": [
            {
                "name": "c",
                "param_type": "required",
                "format": [
                    "ND"
                ],
                "type": [
                    "float"
                ]
            }
        ]
    }
]

### 创建算子工程
根据上文的算子原型json文件，创建算子工程。该部分不需要修改，直接执行即可。

In [ ]:
# 清除已有的custom_op目录
!rm -rf Sources/04.03/custom_op
# 使用msopgen工具创建算子工程前设置使用开源仓编译的msopgen工具，不设置时默认使用CANN安装目录下的msopgen工具
!export PATH=/usr/local/bin:~/.local/bin:$PATH;msopgen gen -i Sources/04.03/MatmulCustom.json -c ai_core-ascend910b1 -lan cpp -out Sources/04.03/custom_op

### 修改Tiling结构体定义
参照课程中MatmulCustom的Tiling结构体定义，修改如下代码。

In [ ]:
%%writefile Sources/04.03/custom_op/op_host/matmul_custom_tiling.h
#include "register/tilingdata_base.h"
#include "tiling/tiling_api.h"
namespace optiling {
BEGIN_TILING_DATA_DEF(MatmulCustomTilingData)
  TILING_DATA_FIELD_DEF(uint32_t, size);
END_TILING_DATA_DEF;

REGISTER_TILING_DATA_CLASS(MatmulCustom, MatmulCustomTilingData)
}

### 修改host侧实现
参照课程中MatmulCustom的host侧实现，修改如下代码。

In [ ]:
%%writefile Sources/04.03/custom_op/op_host/matmul_custom.cpp


### 修改Kernel侧实现
参照课程中MatmulCustom的kernel侧实现，修改如下代码。

In [ ]:
%%writefile Sources/04.03/custom_op/op_kernel/matmul_custom.cpp

#include "kernel_operator.h"
#include "lib/matmul_intf.h"

using namespace matmul;
extern "C" __global__ __aicore__ void matmul_custom(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c, GM_ADDR workspace, GM_ADDR tiling) {
    GET_TILING_DATA(tiling_data, tiling);
    // TODO: user kernel impl
}

### 测试
以下代码为Matmul算子的测试代码，代码进行M = 1024, N = 2048, K = 512的测试。完成该测试后可以尝试修改算子支持其他Shape，并修改该测试代码进行测试。

In [ ]:
%%writefile Sources/04.03/aclnn_test.cpp

#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "aclnn/aclnn_base.h"
#include "aclnn/acl_meta.h"
#include "acl/acl_rt.h"
#include "aclnn_matmul_custom.h"

#define CHECK_ACL(expr)                                                                                 \
    do {                                                                                                \
        auto __ret = (expr);                                                                            \
        int32_t __code = static_cast<int32_t>(__ret);                                                   \
        if (__code != 0) {                                                                              \
            fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__, __code); \
        }                                                                                               \
    } while (0)

int32_t main(int32_t argc, char** argv)
{
    const int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    CHECK_ACL(aclnnInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));

    // 定义MatmulAbs的张量形状
    const std::vector<int64_t> shape_a = {1024, 512};    // 输入a形状
    const std::vector<int64_t> shape_b = {512, 2048};     // 输入b形状
    const std::vector<int64_t> shape_bias = {2048};       // 输入bias形状
    const std::vector<int64_t> shape_output = {1024, 2048};// 输出形状

    // 计算各张量元素数量和内存大小
    const int64_t elementCount_a = shape_a[0] * shape_a[1];
    const int64_t elementCount_b = shape_b[0] * shape_b[1];
    const int64_t elementCount_bias = shape_bias[0];
    const int64_t elementCount_output = shape_output[0] * shape_output[1];
    
    const size_t bufferSize_a = elementCount_a * sizeof(aclFloat16);
    const size_t bufferSize_b = elementCount_b * sizeof(aclFloat16);
    const size_t bufferSize_bias = elementCount_bias * sizeof(float);
    const size_t bufferSize_output = elementCount_output * sizeof(float);

    // 分配输入a的设备内存并创建张量
    void* inputADeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputADeviceMem, bufferSize_a, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputA = aclCreateTensor(shape_a.data(), shape_a.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                        shape_a.data(), shape_a.size(), inputADeviceMem);

    // 分配输入b的设备内存并创建张量
    void* inputBDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputBDeviceMem, bufferSize_b, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputB = aclCreateTensor(shape_b.data(), shape_b.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                        shape_b.data(), shape_b.size(), inputBDeviceMem);

    // 分配输入bias的设备内存并创建张量
    void* inputBiasDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputBiasDeviceMem, bufferSize_bias, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputBias = aclCreateTensor(shape_bias.data(), shape_bias.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                           shape_bias.data(), shape_bias.size(), inputBiasDeviceMem);

    // 分配输出的设备内存并创建张量
    void* outputDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&outputDeviceMem, bufferSize_output, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* output = aclCreateTensor(shape_output.data(), shape_output.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                        shape_output.data(), shape_output.size(), outputDeviceMem);

    // 准备主机测试数据（沿用原始代码的aclFloatToFloat16函数）
    std::vector<aclFloat16> inputAHostData(elementCount_a, aclFloatToFloat16(-1.0));
    std::vector<aclFloat16> inputBHostData(elementCount_b, aclFloatToFloat16(2.0));
    std::vector<float> inputBiasHostData(elementCount_bias, float(0.5));
    std::vector<float> outputHostData(elementCount_output, float(0.0));
    
    // 计算预期结果
    std::vector<float> goldenData(elementCount_output, -1023.5f);

    // 主机数据拷贝到设备
    CHECK_ACL(aclrtMemcpy(inputADeviceMem, bufferSize_a, inputAHostData.data(),
                          bufferSize_a, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBDeviceMem, bufferSize_b, inputBHostData.data(),
                          bufferSize_b, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBiasDeviceMem, bufferSize_bias, inputBiasHostData.data(),
                          bufferSize_bias, ACL_MEMCPY_HOST_TO_DEVICE));

    // 获取Matmul算子工作空间大小和执行器
    uint64_t workspaceSize = 0;
    aclOpExecutor* executor = nullptr;
    CHECK_ACL(aclnnMatmulCustomGetWorkspaceSize(inputA, inputB, inputBias, output, &workspaceSize, &executor));

    // 分配工作空间内存
    void* workspaceDeviceMem = nullptr;
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtMalloc(&workspaceDeviceMem, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST));
    }

    // 执行Matmul算子
    CHECK_ACL(aclnnMatmulCustom(workspaceDeviceMem, workspaceSize, executor, stream));
    CHECK_ACL(aclrtSynchronizeStream(stream));

    // 设备结果拷贝回主机
    CHECK_ACL(aclrtMemcpy(outputHostData.data(), bufferSize_output, outputDeviceMem,
                          bufferSize_output, ACL_MEMCPY_DEVICE_TO_HOST));

    // 打印结果并验证
    printf("result is:\n");
    const int64_t previewCount = std::min<int64_t>(elementCount_output, 10);
    for (int64_t i = 0; i < previewCount; i++) { 
        printf("%.1f ", outputHostData[i]); 
    }
    printf("\ntest %s\n", std::equal(outputHostData.begin(), outputHostData.end(), goldenData.begin()) ? "pass" : "failed");

    // 释放资源
    aclDestroyTensor(inputA);
    aclDestroyTensor(inputB);
    aclDestroyTensor(inputBias);
    aclDestroyTensor(output);
    
    CHECK_ACL(aclrtFree(inputADeviceMem));
    CHECK_ACL(aclrtFree(inputBDeviceMem));
    CHECK_ACL(aclrtFree(inputBiasDeviceMem));
    CHECK_ACL(aclrtFree(outputDeviceMem));
    
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtFree(workspaceDeviceMem));
    }
    
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclnnFinalize());
    
    return 0;
}

修改完成后，重新编译部署算子，运行测试代码，检查输出是否与预期一致。

In [ ]:
#编译算子工程并部署编译出的算子包
!cd Sources/04.03/custom_op;bash build.sh;
!./Sources/04.03/custom_op/build_out/custom_opp*.run --install-path=${HOME}/

In [ ]:
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/04.03/aclnn_test.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/04.03/execute_op;
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;./Sources/04.03/execute_op

预期执行效果如下：

<img src="./images/result_04.png" alt="结果图片04"  width="500px" >

### 执行以下代码获取答案。

host侧实现

In [ ]:
!cat ./answer/04.03_answer/op_host/matmul_custom.cpp

Tiling结构体定义

In [ ]:
!cat ./answer/04.03_answer/op_host/matmul_custom_tiling.h

kernel实现

In [ ]:
!cat ./answer/04.03_answer/op_kernel/matmul_custom.cpp